# Article Writer Agents: a three-agent crew
# Maqola yozuvchi agentlar: uch agentli jamoa

*Corrected version (Chapter 20). The original is kept as `article_writer_tutorial_original.ipynb`.*

A **planner**, a **writer** and an **editor** work one after another (CrewAI's *sequential* process). The notebook runs with CrewAI and a local Ollama model if both are installed; otherwise it runs the same pipeline with the 60-line `crew_mini.py` and a stub model, so that you still see *what each agent receives*.

Planner, yozuvchi va muharrir ketma-ket ishlaydi. CrewAI va Ollama o'rnatilgan bo'lsa, haqiqiy model ishlaydi; aks holda `crew_mini.py` bilan shu jarayonning soddalashtirilgan nusxasi ishlaydi.

In [ ]:
import os
# the original ended with 'HTTPSConnectionPool(host=telemetry.crewai.com ...)': CrewAI sends anonymous
# usage data and fails noisily when offline. Opt out BEFORE importing crewai.
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"

try:
    import crewai
    from crewai import Agent, Task, Crew, Process, LLM
    HAVE_CREWAI = True
except ImportError:
    HAVE_CREWAI = False
print("CrewAI available:", HAVE_CREWAI)

## 1. The language model / Til modeli

Install Ollama (https://ollama.com/download) and run `ollama pull llama3.1` in a terminal. The `LLM` object needs the model name **with the provider prefix** and the address of the Ollama server (`base_url`); without it some versions look for an OpenAI key. The original used `llama3:latest` without `base_url`.

In [ ]:
import socket

def ollama_running(host="localhost", port=11434):
    try:
        socket.create_connection((host, port), timeout=1).close()
        return True
    except OSError:
        return False

USE_REAL_LLM = HAVE_CREWAI and ollama_running()
print("real LLM:", USE_REAL_LLM)

if USE_REAL_LLM:
    llm = LLM(model="ollama/llama3.1", base_url="http://localhost:11434", temperature=0.3)
    # the original asked 'What is the capital of Uzbekistan' but its comment said 'explain CrewAI';
    # a test must test what it says:
    print(llm.call("What is the capital of Uzbekistan? Answer in one word."))

## 2. Agents / Agentlar

`allow_delegation=False` keeps every agent to its own task. The temperature is **0.3**, not 0.7: a plan and an edit should be reproducible; creativity is needed only in the draft. *The planner cannot see the web*, so the original instruction 'prioritize the latest trends and news' invites invention; we tell it to separate known facts from guesses.

In [ ]:
if USE_REAL_LLM:
    def make_agents(llm):
        planner = Agent(role="Content Planner",
            goal="Plan engaging and factually accurate content on {topic}",
            backstory="You plan a blog article about the topic: {topic}. You have NO internet access: "
                      "use only facts you are sure of and mark anything uncertain with '(verify)'. "
                      "Your outline is the basis for the Content Writer.",
            llm=llm, allow_delegation=False, verbose=True)
        writer = Agent(role="Content Writer",
            goal="Write an insightful and factually accurate opinion piece about the topic: {topic}",
            backstory="You write an opinion piece about {topic} from the Content Planner's outline. "
                      "You are objective, you keep the '(verify)' marks, and you say when a statement is an opinion.",
            llm=llm, allow_delegation=False, verbose=True)
        editor = Agent(role="Editor",
            goal="Edit a blog post to align with the writing style of the organization",
            backstory="You receive a blog post from the Content Writer. You check that it follows journalistic "
                      "best practices, gives balanced viewpoints, and avoids major controversial topics.",
            llm=llm, allow_delegation=False, verbose=True)
        return planner, writer, editor
    planner, writer, editor = make_agents(llm)

## 3. Tasks / Vazifalar

In a sequential crew each task receives the output of the previous one as *context*. We state it explicitly with `context=[...]` so the data flow is visible. The editor's task says what to do (the original only said 'proofread').

In [ ]:
if USE_REAL_LLM:
    plan = Task(description="1. List the key points and well-established facts on {topic}.\n"
                            "2. Identify the target audience.\n"
                            "3. Write an outline: introduction, key points, call to action.\n"
                            "4. Suggest SEO keywords. Mark uncertain facts with '(verify)'.",
                expected_output="A content plan: outline, audience analysis, SEO keywords.",
                agent=planner)
    write = Task(description="1. Use the plan to write a blog post on {topic}.\n2. Use the keywords naturally.\n"
                             "3. Give the sections engaging titles; add an introduction and a conclusion.",
                 expected_output="A blog post in markdown; each section has 2 or 3 paragraphs.",
                 agent=writer, context=[plan])
    edit = Task(description="Proofread the post for grammar and style; fix unsupported claims or mark them '(verify)'.",
                expected_output="The final blog post in markdown, ready for publication.",
                agent=editor, context=[write])

## 4. The crew and the run / Jamoa va ishga tushirish

`Process.sequential` is the default; we write it out. The original printed `tasks_output[1]` (the *writer's* draft) after saying 'the final result'; the final text is `result.raw`, the editor's output.

In [ ]:
topic = "Future of AI"

if USE_REAL_LLM:
    crew = Crew(agents=[planner, writer, editor], tasks=[plan, write, edit],
                process=Process.sequential, verbose=False)
    result = crew.kickoff(inputs={"topic": topic})
    final_text = result.raw                       # the editor's output
    draft_text = result.tasks_output[1].raw       # the writer's draft, for comparison
    print("words in draft / final:", len(draft_text.split()), len(final_text.split()))
    print(getattr(result, "token_usage", ""))     # prompt/completion tokens, if the version provides it
else:
    from crew_mini import Agent as A, Task as T, Crew as C_, stub_llm, words
    pl = A("Content Planner", "Plan content on {topic}", "You plan a blog article about {topic}.", stub_llm("plan", 300))
    wr = A("Content Writer", "Write an opinion piece on {topic}", "You write from the planner's outline.", stub_llm("draft", 900))
    ed = A("Editor", "Edit the post", "You review the writer's post.", stub_llm("final", 900))
    t1 = T("Outline an article on {topic}.", "An outline.", pl)
    t2 = T("Write the article on {topic}.", "A blog post.", wr, context=[t1])
    t3 = T("Proofread the post.", "The final post.", ed, context=[t2])
    crew = C_([pl, wr, ed], [t1, t2, t3])
    final_text = crew.kickoff({"topic": topic})
    print("input words per stage:", [words(p) for p in crew.prompts])
    print("the editor's prompt contains the draft:", "[draft]" in crew.prompts[2])

In [ ]:
from IPython.display import Markdown
Markdown(final_text if USE_REAL_LLM else final_text[:200] + " ...")

## 5. What to try / Nimani sinab ko'rish

1. Run the crew twice and diff the outputs: how reproducible is it at temperature 0.3, at 0.9?
2. Remove the `(verify)` instruction and count the unsupported claims in the final text.
3. Add a fourth agent, a fact-checker with a web-search tool, and compare the number of tool calls and tokens.
4. Replace `Process.sequential` by `Process.hierarchical` (this needs `manager_llm`) and compare cost and result.